# Error Analysis: Identify characteristics of misclassified sentences

- For the sentences that were incorrectly classified, what are the characteristics in them?
- For the ones that are classified as a prediction, do our prediction properties exist?

---

## TODOs

1. Majority Vote (mv): to get the ultimate $ y_{hat} $ label
    1. Do models (mv) tend to be correct or incorrect?
2. Fliess Kappa: to measure agreement
    1. Do models tend to agree and be correct or incorrect?
3. K-Means: plotting where

In [1]:
import os
import sys

import numpy as np
import pandas as pd



notebook_dir = os.getcwd()

sys.path.append(os.path.join(notebook_dir, '../'))

from data_processing import DataProcessing
from data_visualizing import DataVisualizing
from feature_extraction import SpacyFeatureExtraction

In [2]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_columns', 40)
# pd.set_option('display.max_rows', None)

## Load Data

In [3]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
# combine_data_path = os.path.join(base_data_path, 'financial_phrase_bank/combined_generated_fin_phrase_bank')
train_data = os.path.join(base_data_path, 'classification_results/train_synthetic-v1_2026-03-23/seed3')

In [4]:
# model_results_path = os.path.join(combine_data_path, 'inference_chronicle2050_2026-03-07_21-51-47.csv')
test_data = os.path.join(train_data, 'external_fpb-maya-binary-imbalanced-96d-v1/', 'ml_classifiers_fpb-maya-binary-imbalanced-96d-v1.csv')

model_results_df = DataProcessing.load_from_file(test_data, 'csv', sep=',')
cols_to_drop = ["Dataset Name", "Author Type", "maya_label"]
compare_y_vs_yhats_df = DataProcessing.drop_df_columns(model_results_df, cols_to_drop)
compare_y_vs_yhats_df

,Base Sentence,Sentence Label,Base Sentence Embedding,perceptron,sgd_classifier,logistic_regression,ridge_classifier,decision_tree_classifier,random_forest_classifier,gradient_boosting_classifier,support_vector_machine_classifier,x_gradient_boosting_classifier
0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,[-1.22568220e-01 2.48624131e-01 -4.82655168e-02 -1.09789923e-01\n -8.58348906e-02 3.63291465e-02 -1.27964811e-02 7.17335939e-02\n -1.18364163e-01 2.27930522e+00 -3.32476139e-01 -1.46164745e-02\n 1.25499114e-01 7.25146085e-02 6.57393187e-02 -5.76901138e-02\n -2.05907249e-03 1.51917112e+00 -3.29480022e-01 -4.57408801e-02\n 2.22120117e-02 1.05339624e-01 -4.22692373e-02 -1.23566784e-01\n -5.79119613e-03 6.18153140e-02 -5.48806489e-02 2.96797305e-02\n -1.07008472e-01 -9.98812243e-02 -9.39770564e-02 3.87728140e-02\n -5.82127571e-02 7.14930072e-02 9.65250358e-02 -3.75642404e-02\n 7.18023349e-03 6.21852428e-02 7.68979266e-02 -1.69069305e-01\n 1.44475931e-03 7.71993026e-02 3.41709815e-02 -1.10992551e-01\n -1.81387588e-02 -8.61631632e-02 -2.34903637e-02 6.00658916e-02\n -3....,1,1,1,1,0,1,1,1,1
1,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth in the range of 20 % -40 % with an operating profit margin of 10 % -20 % of net sales .",1,[-2.63320029e-01 3.28540474e-01 4.48558182e-02 -1.09422831e-02\n 3.39385755e-02 -1.25387803e-01 2.09340192e-02 8.32975954e-02\n 1.10356383e-01 1.74510765e+00 -3.98153275e-01 1.41264707e-01\n -8.98099765e-02 -1.47727029e-02 5.77163734e-02 -5.09223454e-02\n -4.27257568e-02 1.41277170e+00 -2.09747717e-01 1.56779997e-02\n -3.64826852e-03 5.97854443e-02 -7.89400712e-02 -1.39893517e-02\n 1.13595366e-01 9.83026922e-02 -5.14178127e-02 7.06899390e-02\n 2.47105230e-02 1.06609218e-01 -4.19648038e-03 -1.85196903e-02\n -1.48957130e-02 9.90139246e-02 1.10973224e-01 -2.59710420e-02\n -6.08825274e-02 4.48305123e-02 4.36691009e-02 -9.27021950e-02\n 6.36729002e-02 1.98753644e-02 2.10891932e-01 -8.54000822e-02\n 4.67465706e-02 -2.04764325e-02 -3.69948000e-02 1.16207667e-01\n 1....,1,1,0,0,1,1,1,0,1
2,TeliaSonera TLSN said the offer is in line with its strategy to increase its ownership in core business holdings and would strengthen Eesti Telekom 's offering to its customers .,1,[ 1.15080299e-02 1.98225155e-01 1.95062440e-02 -5.97744621e-02\n 1.75585955e-01 -1.25474662e-01 -3.72287892e-02 -2.83286646e-02\n -3.56662273e-02 1.96045303e+00 -2.79843062e-01 -1.04898110e-01\n 4.49508205e-02 1.31481076e-02 6.55795187e-02 -4.11130209e-03\n -5.56710083e-03 1.11790729e+00 -1.21459611e-01 -9.75316912e-02\n 2.18506586e-02 1.38311490e-01 9.17155594e-02 -5.85136563e-02\n 1.03342809e-01 7.32071251e-02 -3.92427929e-02 1.13294438e-01\n -1.07247204e-01 5.58346435e-02 -9.28281173e-02 5.48683619e-03\n -1.84886660e-02 5.24056628e-02 4.05881368e-02 -8.35432187e-02\n 1.13391867e-02 2.78038122e-02 -5.44880610e-03 -1.02035753e-01\n -6.69988245e-02 7.79658780e-02 5.37769347e-02 -1.39616817e-01\n 2.96867881e-02 1.13102477e-02 -3.53476144e-02 -1.88835058e-02\n -4....,1,1,1,1,1,1,1,1,1
3,"STORA ENSO , NORSKE SKOG , M-REAL , UPM-KYMMENE Credit Suisse First Boston ( CFSB ) raised the fair value for shares in four of the largest Nordic forestry groups .",1,[-5.94305294e-03 1.57299116e-01 -1.01663664e-01 -4.74716797e-02\n 1.79840446e-01 2.63532978e-02 4.07044962e-02 -3.17418873e-02\n -1.32127828e-03 1.28309309e+00 -3.01563025e-01 4.19499092e-02\n 4.76229452e-02 -1.16947450e-01 -1.42535036e-02 2.37643998e-02\n -3.57760116e-02 8.38545859e-01 -5.43218441e-02 2.42432952e-03\n 7.67285563e-03 4.88467440e-02 4.74129207e-02 -2.67546187e-04\n 6.47759363e-02 7.66171236e-03 -7.71588413e-03 6.14660196e-02\n 9.25298501e-03 1.29891783e-01 6.76145032e-02 -7.35822739e-03\n 3.22291143e-02 1.50113806e-01 1.32441163e-01 -9.81159508e-03\n -1.3941115

In [5]:
model_col_names = compare_y_vs_yhats_df.columns.to_list()[3:]
model_col_names, len(model_col_names)

(['perceptron',
  'sgd_classifier',
  'logistic_regression',
  'ridge_classifier',
  'decision_tree_classifier',
  'random_forest_classifier',
  'gradient_boosting_classifier',
  'support_vector_machine_classifier',
  'x_gradient_boosting_classifier'],
 9)

In [6]:
compare_y_vs_mv_df = compare_y_vs_yhats_df[model_col_names].mode(axis=1)
compare_y_vs_yhats_df['Majority Vote'] = compare_y_vs_mv_df
compare_y_vs_yhats_df

,Base Sentence,Sentence Label,Base Sentence Embedding,perceptron,sgd_classifier,logistic_regression,ridge_classifier,decision_tree_classifier,random_forest_classifier,gradient_boosting_classifier,support_vector_machine_classifier,x_gradient_boosting_classifier,Majority Vote
0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,[-1.22568220e-01 2.48624131e-01 -4.82655168e-02 -1.09789923e-01\n -8.58348906e-02 3.63291465e-02 -1.27964811e-02 7.17335939e-02\n -1.18364163e-01 2.27930522e+00 -3.32476139e-01 -1.46164745e-02\n 1.25499114e-01 7.25146085e-02 6.57393187e-02 -5.76901138e-02\n -2.05907249e-03 1.51917112e+00 -3.29480022e-01 -4.57408801e-02\n 2.22120117e-02 1.05339624e-01 -4.22692373e-02 -1.23566784e-01\n -5.79119613e-03 6.18153140e-02 -5.48806489e-02 2.96797305e-02\n -1.07008472e-01 -9.98812243e-02 -9.39770564e-02 3.87728140e-02\n -5.82127571e-02 7.14930072e-02 9.65250358e-02 -3.75642404e-02\n 7.18023349e-03 6.21852428e-02 7.68979266e-02 -1.69069305e-01\n 1.44475931e-03 7.71993026e-02 3.41709815e-02 -1.10992551e-01\n -1.81387588e-02 -8.61631632e-02 -2.34903637e-02 6.00658916e-02\n -3....,1,1,1,1,0,1,1,1,1,1
1,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth in the range of 20 % -40 % with an operating profit margin of 10 % -20 % of net sales .",1,[-2.63320029e-01 3.28540474e-01 4.48558182e-02 -1.09422831e-02\n 3.39385755e-02 -1.25387803e-01 2.09340192e-02 8.32975954e-02\n 1.10356383e-01 1.74510765e+00 -3.98153275e-01 1.41264707e-01\n -8.98099765e-02 -1.47727029e-02 5.77163734e-02 -5.09223454e-02\n -4.27257568e-02 1.41277170e+00 -2.09747717e-01 1.56779997e-02\n -3.64826852e-03 5.97854443e-02 -7.89400712e-02 -1.39893517e-02\n 1.13595366e-01 9.83026922e-02 -5.14178127e-02 7.06899390e-02\n 2.47105230e-02 1.06609218e-01 -4.19648038e-03 -1.85196903e-02\n -1.48957130e-02 9.90139246e-02 1.10973224e-01 -2.59710420e-02\n -6.08825274e-02 4.48305123e-02 4.36691009e-02 -9.27021950e-02\n 6.36729002e-02 1.98753644e-02 2.10891932e-01 -8.54000822e-02\n 4.67465706e-02 -2.04764325e-02 -3.69948000e-02 1.16207667e-01\n 1....,1,1,0,0,1,1,1,0,1,1
2,TeliaSonera TLSN said the offer is in line with its strategy to increase its ownership in core business holdings and would strengthen Eesti Telekom 's offering to its customers .,1,[ 1.15080299e-02 1.98225155e-01 1.95062440e-02 -5.97744621e-02\n 1.75585955e-01 -1.25474662e-01 -3.72287892e-02 -2.83286646e-02\n -3.56662273e-02 1.96045303e+00 -2.79843062e-01 -1.04898110e-01\n 4.49508205e-02 1.31481076e-02 6.55795187e-02 -4.11130209e-03\n -5.56710083e-03 1.11790729e+00 -1.21459611e-01 -9.75316912e-02\n 2.18506586e-02 1.38311490e-01 9.17155594e-02 -5.85136563e-02\n 1.03342809e-01 7.32071251e-02 -3.92427929e-02 1.13294438e-01\n -1.07247204e-01 5.58346435e-02 -9.28281173e-02 5.48683619e-03\n -1.84886660e-02 5.24056628e-02 4.05881368e-02 -8.35432187e-02\n 1.13391867e-02 2.78038122e-02 -5.44880610e-03 -1.02035753e-01\n -6.69988245e-02 7.79658780e-02 5.37769347e-02 -1.39616817e-01\n 2.96867881e-02 1.13102477e-02 -3.53476144e-02 -1.88835058e-02\n -4....,1,1,1,1,1,1,1,1,1,1
3,"STORA ENSO , NORSKE SKOG , M-REAL , UPM-KYMMENE Credit Suisse First Boston ( CFSB ) raised the fair value for shares in four of the largest Nordic forestry groups .",1,[-5.94305294e-03 1.57299116e-01 -1.01663664e-01 -4.74716797e-02\n 1.79840446e-01 2.63532978e-02 4.07044962e-02 -3.17418873e-02\n -1.32127828e-03 1.28309309e+00 -3.01563025e-01 4.19499092e-02\n 4.76229452e-02 -1.16947450e-01 -1.42535036e-02 2.37643998e-02\n -3.57760116e-02 8.38545859e-01 -5.43218441e-02 2.42432952e-03\n 7.67285563e-03 4.88467440e-02 4.74129207e-02 -2.67546187e-04\n 6.47759363e-02 7.66171236e-03 -7.71588413e-03 6.14660196e-02\n 9.25298501e-03 1.29891783e-01 6.76145032e-02 -7.35822739e-03\n 3.22291143e-02 1.50113806e-01 1.32441163e-01 -9.81159

In [7]:
model_col_names

['perceptron',
 'sgd_classifier',
 'logistic_regression',
 'ridge_classifier',
 'decision_tree_classifier',
 'random_forest_classifier',
 'gradient_boosting_classifier',
 'support_vector_machine_classifier',
 'x_gradient_boosting_classifier']

In [8]:
meta_data = compare_y_vs_yhats_df.iloc[:, 2:-1]
meta_data

,Base Sentence Embedding,perceptron,sgd_classifier,logistic_regression,ridge_classifier,decision_tree_classifier,random_forest_classifier,gradient_boosting_classifier,support_vector_machine_classifier,x_gradient_boosting_classifier
0,[-1.22568220e-01 2.48624131e-01 -4.82655168e-02 -1.09789923e-01\n -8.58348906e-02 3.63291465e-02 -1.27964811e-02 7.17335939e-02\n -1.18364163e-01 2.27930522e+00 -3.32476139e-01 -1.46164745e-02\n 1.25499114e-01 7.25146085e-02 6.57393187e-02 -5.76901138e-02\n -2.05907249e-03 1.51917112e+00 -3.29480022e-01 -4.57408801e-02\n 2.22120117e-02 1.05339624e-01 -4.22692373e-02 -1.23566784e-01\n -5.79119613e-03 6.18153140e-02 -5.48806489e-02 2.96797305e-02\n -1.07008472e-01 -9.98812243e-02 -9.39770564e-02 3.87728140e-02\n -5.82127571e-02 7.14930072e-02 9.65250358e-02 -3.75642404e-02\n 7.18023349e-03 6.21852428e-02 7.68979266e-02 -1.69069305e-01\n 1.44475931e-03 7.71993026e-02 3.41709815e-02 -1.10992551e-01\n -1.81387588e-02 -8.61631632e-02 -2.34903637e-02 6.00658916e-02\n -3....,1,1,1,1,0,1,1,1,1
1,[-2.63320029e-01 3.28540474e-01 4.48558182e-02 -1.09422831e-02\n 3.39385755e-02 -1.25387803e-01 2.09340192e-02 8.32975954e-02\n 1.10356383e-01 1.74510765e+00 -3.98153275e-01 1.41264707e-01\n -8.98099765e-02 -1.47727029e-02 5.77163734e-02 -5.09223454e-02\n -4.27257568e-02 1.41277170e+00 -2.09747717e-01 1.56779997e-02\n -3.64826852e-03 5.97854443e-02 -7.89400712e-02 -1.39893517e-02\n 1.13595366e-01 9.83026922e-02 -5.14178127e-02 7.06899390e-02\n 2.47105230e-02 1.06609218e-01 -4.19648038e-03 -1.85196903e-02\n -1.48957130e-02 9.90139246e-02 1.10973224e-01 -2.59710420e-02\n -6.08825274e-02 4.48305123e-02 4.36691009e-02 -9.27021950e-02\n 6.36729002e-02 1.98753644e-02 2.10891932e-01 -8.54000822e-02\n 4.67465706e-02 -2.04764325e-02 -3.69948000e-02 1.16207667e-01\n 1....,1,1,0,0,1,1,1,0,1
2,[ 1.15080299e-02 1.98225155e-01 1.95062440e-02 -5.97744621e-02\n 1.75585955e-01 -1.25474662e-01 -3.72287892e-02 -2.83286646e-02\n -3.56662273e-02 1.96045303e+00 -2.79843062e-01 -1.04898110e-01\n 4.49508205e-02 1.31481076e-02 6.55795187e-02 -4.11130209e-03\n -5.56710083e-03 1.11790729e+00 -1.21459611e-01 -9.75316912e-02\n 2.18506586e-02 1.38311490e-01 9.17155594e-02 -5.85136563e-02\n 1.03342809e-01 7.32071251e-02 -3.92427929e-02 1.13294438e-01\n -1.07247204e-01 5.58346435e-02 -9.28281173e-02 5.48683619e-03\n -1.84886660e-02 5.24056628e-02 4.05881368e-02 -8.35432187e-02\n 1.13391867e-02 2.78038122e-02 -5.44880610e-03 -1.02035753e-01\n -6.69988245e-02 7.79658780e-02 5.37769347e-02 -1.39616817e-01\n 2.96867881e-02 1.13102477e-02 -3.53476144e-02 -1.88835058e-02\n -4....,1,1,1,1,1,1,1,1,1
3,[-5.94305294e-03 1.57299116e-01 -1.01663664e-01 -4.74716797e-02\n 1.79840446e-01 2.63532978e-02 4.07044962e-02 -3.17418873e-02\n -1.32127828e-03 1.28309309e+00 -3.01563025e-01 4.19499092e-02\n 4.76229452e-02 -1.16947450e-01 -1.42535036e-02 2.37643998e-02\n -3.57760116e-02 8.38545859e-01 -5.43218441e-02 2.42432952e-03\n 7.67285563e-03 4.88467440e-02 4.74129207e-02 -2.67546187e-04\n 6.47759363e-02 7.66171236e-03 -7.71588413e-03 6.14660196e-02\n 9.25298501e-03 1.29891783e-01 6.76145032e-02 -7.35822739e-03\n 3.22291143e-02 1.50113806e-01 1.32441163e-01 -9.81159508e-03\n -1.39411157e-02 1.04931206e-03 -1.90248825e-02 -9.27351974e-03\n -2.62032263e-02 7.86117762e-02 8.33134279e-02 2.64521260e-02\n -1.58701912e-02 9.79095325e-03 3.53994928e-02 -1.72072705e-02\n 7....,0,0,0,0,1,0,0,0,0
4,[-9.75671411e-02 2.17742935e-01 -5.24893478e-02 2.29859371e-02\n 1.95262451e-02 -3.96593288e-02 -2.93270722e-02 1.45730842e-02\n 8.01267326e-02 2.14558053e+00 -3.02207738e-01 7.73812085e-02\n 9.75400433e-02 -5.86019419e-02 2.85570417e-02 -6.01695329e-02\n -6.48141513e-03 1.10976875e+00 -1.62209153e-01 -2.63800696e-02\n 6.90186918e-02 5.40496930e-02 -4.89046909e-02 -1.06065851e-02\n 3.81638743e-02 8.63624960e-02 -1.46540195e-01 -7.51399994e-02\n -5.42312972e-02 -2.43689902e-02 3.15490589e-02 -7.77387097e-02\n -1.04405805e-01 1.55526847e-01 2.55947262e-02 -2.01004483e-02\n -4.3354682

In [9]:
meta_data = compare_y_vs_yhats_df.iloc[:, 3:-1]
compare_y_vs_yhats_df['Meta Data'] = meta_data.to_dict(orient='records')
compare_y_vs_yhats_df
y_vs_yhats_df = DataProcessing.drop_df_columns(compare_y_vs_yhats_df, meta_data.columns.to_list())
y_vs_yhats_df


,Base Sentence,Sentence Label,Base Sentence Embedding,Majority Vote,Meta Data
0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,[-1.22568220e-01 2.48624131e-01 -4.82655168e-02 -1.09789923e-01\n -8.58348906e-02 3.63291465e-02 -1.27964811e-02 7.17335939e-02\n -1.18364163e-01 2.27930522e+00 -3.32476139e-01 -1.46164745e-02\n 1.25499114e-01 7.25146085e-02 6.57393187e-02 -5.76901138e-02\n -2.05907249e-03 1.51917112e+00 -3.29480022e-01 -4.57408801e-02\n 2.22120117e-02 1.05339624e-01 -4.22692373e-02 -1.23566784e-01\n -5.79119613e-03 6.18153140e-02 -5.48806489e-02 2.96797305e-02\n -1.07008472e-01 -9.98812243e-02 -9.39770564e-02 3.87728140e-02\n -5.82127571e-02 7.14930072e-02 9.65250358e-02 -3.75642404e-02\n 7.18023349e-03 6.21852428e-02 7.68979266e-02 -1.69069305e-01\n 1.44475931e-03 7.71993026e-02 3.41709815e-02 -1.10992551e-01\n -1.81387588e-02 -8.61631632e-02 -2.34903637e-02 6.00658916e-02\n -3....,1,"{'perceptron': 1, 'sgd_classifier': 1, 'logistic_regression': 1, 'ridge_classifier': 1, 'decision_tree_classifier': 0, 'random_forest_classifier': 1, 'gradient_boosting_classifier': 1, 'support_vector_machine_classifier': 1, 'x_gradient_boosting_classifier': 1}"
1,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth in the range of 20 % -40 % with an operating profit margin of 10 % -20 % of net sales .",1,[-2.63320029e-01 3.28540474e-01 4.48558182e-02 -1.09422831e-02\n 3.39385755e-02 -1.25387803e-01 2.09340192e-02 8.32975954e-02\n 1.10356383e-01 1.74510765e+00 -3.98153275e-01 1.41264707e-01\n -8.98099765e-02 -1.47727029e-02 5.77163734e-02 -5.09223454e-02\n -4.27257568e-02 1.41277170e+00 -2.09747717e-01 1.56779997e-02\n -3.64826852e-03 5.97854443e-02 -7.89400712e-02 -1.39893517e-02\n 1.13595366e-01 9.83026922e-02 -5.14178127e-02 7.06899390e-02\n 2.47105230e-02 1.06609218e-01 -4.19648038e-03 -1.85196903e-02\n -1.48957130e-02 9.90139246e-02 1.10973224e-01 -2.59710420e-02\n -6.08825274e-02 4.48305123e-02 4.36691009e-02 -9.27021950e-02\n 6.36729002e-02 1.98753644e-02 2.10891932e-01 -8.54000822e-02\n 4.67465706e-02 -2.04764325e-02 -3.69948000e-02 1.16207667e-01\n 1....,1,"{'perceptron': 1, 'sgd_classifier': 1, 'logistic_regression': 0, 'ridge_classifier': 0, 'decision_tree_classifier': 1, 'random_forest_classifier': 1, 'gradient_boosting_classifier': 1, 'support_vector_machine_classifier': 0, 'x_gradient_boosting_classifier': 1}"
2,TeliaSonera TLSN said the offer is in line with its strategy to increase its ownership in core business holdings and would strengthen Eesti Telekom 's offering to its customers .,1,[ 1.15080299e-02 1.98225155e-01 1.95062440e-02 -5.97744621e-02\n 1.75585955e-01 -1.25474662e-01 -3.72287892e-02 -2.83286646e-02\n -3.56662273e-02 1.96045303e+00 -2.79843062e-01 -1.04898110e-01\n 4.49508205e-02 1.31481076e-02 6.55795187e-02 -4.11130209e-03\n -5.56710083e-03 1.11790729e+00 -1.21459611e-01 -9.75316912e-02\n 2.18506586e-02 1.38311490e-01 9.17155594e-02 -5.85136563e-02\n 1.03342809e-01 7.32071251e-02 -3.92427929e-02 1.13294438e-01\n -1.07247204e-01 5.58346435e-02 -9.28281173e-02 5.48683619e-03\n -1.84886660e-02 5.24056628e-02 4.05881368e-02 -8.35432187e-02\n 1.13391867e-02 2.78038122e-02 -5.44880610e-03 -1.02035753e-01\n -6.69988245e-02 7.79658780e-02 5.37769347e-02 -1.39616817e-01\n 2.96867881e-02 1.13102477e-02 -3.53476144e-02 -1.88835058e-02\n -4....,1,"{'perceptron': 1, 'sgd_classifier': 1, 'logistic_regression': 1, 'ridge_classifier': 1, 'decision_tree_classifier': 1, 'random_forest_classifier': 1, 'gradient_boosting_classifier': 1, 'support_vector_machine_classifier': 1, 'x_gradient_boosting_classifier': 1}"
3,"STORA ENSO , NORSKE SKOG , M-REAL , UPM-KYMMENE Credit Suisse First Boston ( CFSB ) raised the fair value for shares in four of the largest Nordic forestry groups .",1,[-5.94305294e-03 1.572

## Get class of each sentence

- [ ] True Label vs Majority Vote
- [ ] True Label vs Each Model

## True Label vs Majority Vote

In [10]:
def classification_of_mv_sentence(df):
    copy_df = df.copy()
    copy_df['Class'] = None

    for idx, row in copy_df.iterrows():
        y = row['Sentence Label']
        y_hat = row['Majority Vote']

        # y = 1 (positive class)
        if y == 1:
            if y_hat == 1:  # TP
                copy_df.loc[idx, 'Class'] = "TP"
            else:  # FN
                copy_df.loc[idx, 'Class'] = "FN"

        # y = 0 (negative class)
        elif y == 0:
            if y_hat == 0:  # TN
                copy_df.loc[idx, 'Class'] = "TN"
            else:  # FP
                copy_df.loc[idx, 'Class'] = "FP"

    return copy_df

In [11]:
mv_with_class_df = classification_of_mv_sentence(y_vs_yhats_df)
mv_with_class_df

,Base Sentence,Sentence Label,Base Sentence Embedding,Majority Vote,Meta Data,Class
0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,[-1.22568220e-01 2.48624131e-01 -4.82655168e-02 -1.09789923e-01\n -8.58348906e-02 3.63291465e-02 -1.27964811e-02 7.17335939e-02\n -1.18364163e-01 2.27930522e+00 -3.32476139e-01 -1.46164745e-02\n 1.25499114e-01 7.25146085e-02 6.57393187e-02 -5.76901138e-02\n -2.05907249e-03 1.51917112e+00 -3.29480022e-01 -4.57408801e-02\n 2.22120117e-02 1.05339624e-01 -4.22692373e-02 -1.23566784e-01\n -5.79119613e-03 6.18153140e-02 -5.48806489e-02 2.96797305e-02\n -1.07008472e-01 -9.98812243e-02 -9.39770564e-02 3.87728140e-02\n -5.82127571e-02 7.14930072e-02 9.65250358e-02 -3.75642404e-02\n 7.18023349e-03 6.21852428e-02 7.68979266e-02 -1.69069305e-01\n 1.44475931e-03 7.71993026e-02 3.41709815e-02 -1.10992551e-01\n -1.81387588e-02 -8.61631632e-02 -2.34903637e-02 6.00658916e-02\n -3....,1,"{'perceptron': 1, 'sgd_classifier': 1, 'logistic_regression': 1, 'ridge_classifier': 1, 'decision_tree_classifier': 0, 'random_forest_classifier': 1, 'gradient_boosting_classifier': 1, 'support_vector_machine_classifier': 1, 'x_gradient_boosting_classifier': 1}",TP
1,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth in the range of 20 % -40 % with an operating profit margin of 10 % -20 % of net sales .",1,[-2.63320029e-01 3.28540474e-01 4.48558182e-02 -1.09422831e-02\n 3.39385755e-02 -1.25387803e-01 2.09340192e-02 8.32975954e-02\n 1.10356383e-01 1.74510765e+00 -3.98153275e-01 1.41264707e-01\n -8.98099765e-02 -1.47727029e-02 5.77163734e-02 -5.09223454e-02\n -4.27257568e-02 1.41277170e+00 -2.09747717e-01 1.56779997e-02\n -3.64826852e-03 5.97854443e-02 -7.89400712e-02 -1.39893517e-02\n 1.13595366e-01 9.83026922e-02 -5.14178127e-02 7.06899390e-02\n 2.47105230e-02 1.06609218e-01 -4.19648038e-03 -1.85196903e-02\n -1.48957130e-02 9.90139246e-02 1.10973224e-01 -2.59710420e-02\n -6.08825274e-02 4.48305123e-02 4.36691009e-02 -9.27021950e-02\n 6.36729002e-02 1.98753644e-02 2.10891932e-01 -8.54000822e-02\n 4.67465706e-02 -2.04764325e-02 -3.69948000e-02 1.16207667e-01\n 1....,1,"{'perceptron': 1, 'sgd_classifier': 1, 'logistic_regression': 0, 'ridge_classifier': 0, 'decision_tree_classifier': 1, 'random_forest_classifier': 1, 'gradient_boosting_classifier': 1, 'support_vector_machine_classifier': 0, 'x_gradient_boosting_classifier': 1}",TP
2,TeliaSonera TLSN said the offer is in line with its strategy to increase its ownership in core business holdings and would strengthen Eesti Telekom 's offering to its customers .,1,[ 1.15080299e-02 1.98225155e-01 1.95062440e-02 -5.97744621e-02\n 1.75585955e-01 -1.25474662e-01 -3.72287892e-02 -2.83286646e-02\n -3.56662273e-02 1.96045303e+00 -2.79843062e-01 -1.04898110e-01\n 4.49508205e-02 1.31481076e-02 6.55795187e-02 -4.11130209e-03\n -5.56710083e-03 1.11790729e+00 -1.21459611e-01 -9.75316912e-02\n 2.18506586e-02 1.38311490e-01 9.17155594e-02 -5.85136563e-02\n 1.03342809e-01 7.32071251e-02 -3.92427929e-02 1.13294438e-01\n -1.07247204e-01 5.58346435e-02 -9.28281173e-02 5.48683619e-03\n -1.84886660e-02 5.24056628e-02 4.05881368e-02 -8.35432187e-02\n 1.13391867e-02 2.78038122e-02 -5.44880610e-03 -1.02035753e-01\n -6.69988245e-02 7.79658780e-02 5.37769347e-02 -1.39616817e-01\n 2.96867881e-02 1.13102477e-02 -3.53476144e-02 -1.88835058e-02\n -4....,1,"{'perceptron': 1, 'sgd_classifier': 1, 'logistic_regression': 1, 'ridge_classifier': 1, 'decision_tree_classifier': 1, 'random_forest_classifier': 1, 'gradient_boosting_classifier': 1, 'support_vector_machine_classifier': 1, 'x_gradient_boosting_classifier': 1}",TP
3,"STORA ENSO , NORSKE SKOG , M-REAL , UPM-KYMMENE Credit Suisse First Boston ( CFSB ) raised the fair value for shares in four of the largest Nordic forestry groups .",1,[-5.943

| $ y $ | $ y_{hat} $ | Outcome |
|----------------|----------------|----------|
| 1              | 1              | **TP**   |
| 1              | 0              | **FN**  (miss out) |
| 0              | 0              | **TN**   |
| 0              | 1              | **FP**  (inflate) |

**Confusion Matrix (sklearn-style layout)**

| y \ y_hat |   0   |   1   |
|-----------|-------|-------|
| **0**     |  TN   |  FP  (inflate) |
| **1**     |  FN  (miss out) |  TP   |

## Linguistic Features

In [12]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_columns', 40)
# pd.set_option('display.max_rows', None)

In [13]:
filt_tp = (mv_with_class_df['Class'] == 'TP')
tp_mv_df = mv_with_class_df[filt_tp]
tp_mv_df.head(3)

,Base Sentence,Sentence Label,Base Sentence Embedding,Majority Vote,Meta Data,Class
0,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,1,[-1.22568220e-01 2.48624131e-01 -4.82655168e-02 -1.09789923e-01\n -8.58348906e-02 3.63291465e-02 -1.27964811e-02 7.17335939e-02\n -1.18364163e-01 2.27930522e+00 -3.32476139e-01 -1.46164745e-02\n 1.25499114e-01 7.25146085e-02 6.57393187e-02 -5.76901138e-02\n -2.05907249e-03 1.51917112e+00 -3.29480022e-01 -4.57408801e-02\n 2.22120117e-02 1.05339624e-01 -4.22692373e-02 -1.23566784e-01\n -5.79119613e-03 6.18153140e-02 -5.48806489e-02 2.96797305e-02\n -1.07008472e-01 -9.98812243e-02 -9.39770564e-02 3.87728140e-02\n -5.82127571e-02 7.14930072e-02 9.65250358e-02 -3.75642404e-02\n 7.18023349e-03 6.21852428e-02 7.68979266e-02 -1.69069305e-01\n 1.44475931e-03 7.71993026e-02 3.41709815e-02 -1.10992551e-01\n -1.81387588e-02 -8.61631632e-02 -2.34903637e-02 6.00658916e-02\n -3....,1,"{'perceptron': 1, 'sgd_classifier': 1, 'logistic_regression': 1, 'ridge_classifier': 1, 'decision_tree_classifier': 0, 'random_forest_classifier': 1, 'gradient_boosting_classifier': 1, 'support_vector_machine_classifier': 1, 'x_gradient_boosting_classifier': 1}",TP
1,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth in the range of 20 % -40 % with an operating profit margin of 10 % -20 % of net sales .",1,[-2.63320029e-01 3.28540474e-01 4.48558182e-02 -1.09422831e-02\n 3.39385755e-02 -1.25387803e-01 2.09340192e-02 8.32975954e-02\n 1.10356383e-01 1.74510765e+00 -3.98153275e-01 1.41264707e-01\n -8.98099765e-02 -1.47727029e-02 5.77163734e-02 -5.09223454e-02\n -4.27257568e-02 1.41277170e+00 -2.09747717e-01 1.56779997e-02\n -3.64826852e-03 5.97854443e-02 -7.89400712e-02 -1.39893517e-02\n 1.13595366e-01 9.83026922e-02 -5.14178127e-02 7.06899390e-02\n 2.47105230e-02 1.06609218e-01 -4.19648038e-03 -1.85196903e-02\n -1.48957130e-02 9.90139246e-02 1.10973224e-01 -2.59710420e-02\n -6.08825274e-02 4.48305123e-02 4.36691009e-02 -9.27021950e-02\n 6.36729002e-02 1.98753644e-02 2.10891932e-01 -8.54000822e-02\n 4.67465706e-02 -2.04764325e-02 -3.69948000e-02 1.16207667e-01\n 1....,1,"{'perceptron': 1, 'sgd_classifier': 1, 'logistic_regression': 0, 'ridge_classifier': 0, 'decision_tree_classifier': 1, 'random_forest_classifier': 1, 'gradient_boosting_classifier': 1, 'support_vector_machine_classifier': 0, 'x_gradient_boosting_classifier': 1}",TP
2,TeliaSonera TLSN said the offer is in line with its strategy to increase its ownership in core business holdings and would strengthen Eesti Telekom 's offering to its customers .,1,[ 1.15080299e-02 1.98225155e-01 1.95062440e-02 -5.97744621e-02\n 1.75585955e-01 -1.25474662e-01 -3.72287892e-02 -2.83286646e-02\n -3.56662273e-02 1.96045303e+00 -2.79843062e-01 -1.04898110e-01\n 4.49508205e-02 1.31481076e-02 6.55795187e-02 -4.11130209e-03\n -5.56710083e-03 1.11790729e+00 -1.21459611e-01 -9.75316912e-02\n 2.18506586e-02 1.38311490e-01 9.17155594e-02 -5.85136563e-02\n 1.03342809e-01 7.32071251e-02 -3.92427929e-02 1.13294438e-01\n -1.07247204e-01 5.58346435e-02 -9.28281173e-02 5.48683619e-03\n -1.84886660e-02 5.24056628e-02 4.05881368e-02 -8.35432187e-02\n 1.13391867e-02 2.78038122e-02 -5.44880610e-03 -1.02035753e-01\n -6.69988245e-02 7.79658780e-02 5.37769347e-02 -1.39616817e-01\n 2.96867881e-02 1.13102477e-02 -3.53476144e-02 -1.88835058e-02\n -4....,1,"{'perceptron': 1, 'sgd_classifier': 1, 'logistic_regression': 1, 'ridge_classifier': 1, 'decision_tree_classifier': 1, 'random_forest_classifier': 1, 'gradient_boosting_classifier': 1, 'support_vector_machine_classifier': 1, 'x_gradient_boosting_classifier': 1}",TP


### Structures of a prediction

|  |   Investigation of Future Reference Expression in Trend Info by Nakajima et al., (2014)   |   spaCy   |
|-----------|-------|-------|
| 1     |  [Action]*[State Change]   |  AUX + VERB |

In [19]:
class_for_strucuture = []
noun_aux_verb_sentences = []
disable_components = []
class_names = mv_with_class_df['Class'].unique()
for class_name in class_names:
    print(f"CLASS: {class_name}")
    filt_class_name = (mv_with_class_df['Class'] == class_name)
    class_df = mv_with_class_df[filt_class_name]
    # print(class_df)
    spacy_fe = SpacyFeatureExtraction(class_df, 'Base Sentence')
    # ner_df = spacy_fe.extract_ner_features(disable_components, visualize=False)
    pos_df = spacy_fe.extract_pos_features(disable_components, visualize=False)
    print(pos_df['Term'].value_counts())
    print(pos_df['Dependency'].value_counts())
    print(pos_df['POS Label'].value_counts())

    # Get sentences that contain NOUN + AUX + VERB
    # noun_sentences = set(pos_df[pos_df['POS Label'] == 'NOUN']['Sentence'].tolist())
    aux_sentences = set(pos_df[pos_df['POS Label'] == 'AUX']['Sentence'].tolist())
    verb_sentences = set(pos_df[pos_df['POS Label'] == 'VERB']['Sentence'].tolist())

    # Sentences that contain all three POS tags
    # class_noun_aux_verb_sentences = noun_sentences & aux_sentences & verb_sentences
    class_noun_aux_verb_sentences = aux_sentences & verb_sentences

    # Track class name and sentences
    for sentence in class_noun_aux_verb_sentences:
        noun_aux_verb_sentences.append({
            'Class': class_name,
            'Sentence': sentence
        })

    print(f"\nCLASS: {class_name}")
    print(f"Number of sentences with NOUN + AUX + VERB: {len(class_noun_aux_verb_sentences)}")
    for sentence in list(class_noun_aux_verb_sentences)[:3]:
        print(f"  - {sentence}")
    print()


CLASS: TP

EXTRACTING POS FEATURES


Extracting POS Features: 100%|██████████| 388/388 [00:01<00:00, 328.05doc/s]



DONE EXTRACTING POS FEATURES
Term
the           399
              388
.             385
to            280
,             251
             ... 
Kalmar          1
Capitex         1
newspapers      1
Ignatius        1
RESULTS         1
Name: count, Length: 2099, dtype: int64
Dependency
prep         905
pobj         879
punct        823
det          692
compound     610
amod         547
nsubj        481
aux          473
ROOT         396
             388
dobj         305
nummod       222
advmod       206
conj         185
poss         165
cc           163
auxpass      151
xcomp        129
ccomp        127
nsubjpass    122
case          84
npadvmod      72
acomp         69
mark          62
advcl         57
nmod          55
quantmod      55
appos         52
relcl         47
attr          38
acl           31
pcomp         27
prt           17
agent         15
neg            9
dep            7
dative         4
csubj          3
expl           3
preconj        3
oprd           3
parataxis      2
pr

Extracting POS Features: 100%|██████████| 70/70 [00:00<00:00, 285.61doc/s]



DONE EXTRACTING POS FEATURES
Term
the             95
,               71
                70
.               70
of              61
                ..
Innova           1
hotel            1
luxury           1
star             1
representing     1
Name: count, Length: 709, dtype: int64
Dependency
prep         224
pobj         219
punct        174
compound     167
det          142
amod         119
nsubj         78
ROOT          71
              70
nummod        64
dobj          57
aux           55
conj          44
cc            38
advmod        23
appos         23
poss          22
nmod          21
auxpass       21
nsubjpass     18
ccomp         17
quantmod      16
xcomp         14
npadvmod      11
advcl          8
acl            8
case           7
relcl          6
mark           4
acomp          4
attr           3
pcomp          3
agent          2
dep            2
prt            2
neg            1
oprd           1
parataxis      1
csubj          1
dative         1
Name: count, dtype: int64


Extracting POS Features: 100%|██████████| 1852/1852 [00:06<00:00, 307.58doc/s]



DONE EXTRACTING POS FEATURES
Term
            1852
.           1831
the         1760
,           1351
to          1200
            ... 
Backup         1
protects       1
stored         1
Higher         1
posting        1
Name: count, Length: 6821, dtype: int64
Dependency
punct        4870
compound     4362
prep         4263
pobj         4161
det          3405
amod         2570
nsubj        2401
ROOT         1921
             1852
dobj         1754
aux          1750
conj         1249
cc           1069
advmod       1003
nummod        931
poss          841
ccomp         520
appos         519
auxpass       462
xcomp         449
case          429
nsubjpass     400
nmod          368
npadvmod      358
acl           287
attr          279
relcl         277
advcl         262
mark          251
acomp         223
pcomp         155
quantmod      147
prt            88
neg            86
agent          80
dep            40
dative         37
predet         18
csubj          16
expl           15
oprd   

Extracting POS Features: 100%|██████████| 2536/2536 [00:08<00:00, 296.17doc/s]



DONE EXTRACTING POS FEATURES
Term
,                      3011
                       2536
.                      2533
the                    2461
of                     1853
                       ... 
www.upm-kymmene.com       1
thirty                    1
seventeen                 1
Free                      1
241.1                     1
Name: count, Length: 8761, dtype: int64
Dependency
pobj         7801
prep         7778
punct        7591
compound     7136
det          4300
amod         3843
nummod       3033
nsubj        2998
ROOT         2684
             2536
conj         2298
dobj         1724
cc           1620
nmod         1168
appos        1086
advmod       1066
poss          970
npadvmod      810
aux           759
case          562
auxpass       418
ccomp         393
nsubjpass     386
attr          371
advcl         352
acl           289
mark          242
quantmod      231
relcl         184
pcomp         136
agent         124
prt           107
acomp         105
dep         

In [20]:
pos_df

,Sentence,Term,Lemma,POS Label,Detailed POS Label,Dependency,Shape,Is Alpha,Stop Word,Unique POS Label,...,Definite,Degree,SubCat,Animacy,Acc,Gen,Loc,Ins,Parat,Prep
0,"The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .",The,the,DET,DT,det,Xxx,True,True,DET_1,...,[Def],,,,,,,,,
1,"The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .",international,international,ADJ,JJ,amod,xxxx,True,False,ADJ_1,...,,[Pos],,,,,,,,
2,"The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .",electronic,electronic,ADJ,JJ,amod,xxxx,True,False,ADJ_2,...,,[Pos],,,,,,,,
3,"The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .",industry,industry,NOUN,NN,compound,xxxx,True,False,NOUN_1,...,,,,,,,,,,
4,"The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .",company,company,NOUN,NN,nsubj,xxxx,True,False,NOUN_2,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65388,"Sales in Finland decreased by 10.5 % in January , while sales outside Finland dropped by 17 % .",by,by,ADP,IN,prep,xx,True,True,ADP_7881,...,,,,,,,,,,
65389,"Sales in Finland decreased by 10.5 % in January , while sales outside Finland dropped by 17 % .",17,17,NUM,CD,nummod,dd,False,False,NUM_4800,...,,,,,,,,,,
65390,"Sales in Finland decreased by 10.5 % in January , while sales outside Finland dropped by 17 % .",%,%,NOUN,NN,pobj,%,False,False,NOUN_13446,...,,,,,,,,,,
65391,"Sales in Finland decreased by 10.5 % in January , while sales outside Finland dropped by 17 % .",.,.,PUNCT,.,punct,.,False,False,PUNCT_7501,...,,,,,,,,,,


In [21]:
# Create dataframe
noun_aux_verb_df = pd.DataFrame(noun_aux_verb_sentences)
noun_aux_verb_df

,Class,Sentence
0,TP,"Known as Post Bank , the concept would see Fidelity Bank rolling out 75 offices in Ghana Post premises , to provide financial services to the people ."
1,TP,"Via the plant , the Belgian company will feed 8.4 MW of electricity into the distribution grid and will be able to deliver up to 14 MW of heat ."
2,TP,Castecka said the town hall would hold talks with other investors interested in the zone .
3,TP,"Jun. 14 , 2009 ( AOL Weblogs delivered by Newstex ) -- Looks like the E71 is about to be upstaged as Nokia 's ( NYSE : NOK ) premier business-class smartphone -- someone in Espoo 's just hit the corporate YouTube account with this promo video for an E72 ."
4,TP,Honkarakenne Oyj - a world-leading manufacturer of genuine wooden homes - will be sponsoring Finnish crosscountry skier Virpi Kuitunen for the next three years .
...,...,...
2729,TN,The Diameter Protocol is developed according to the standards IETF RFC 3588 and IETF RFC 3539 .
2730,TN,"Of the company 's net sales , 38 % was acquired in Finland , 21 % in other European countries , 40 % in Asia , and 1 % in the US ."
2731,TN,"Outotec , headquartered in Espoo , Finland , is a leading provider of process solutions , technologies and services for the mining and metallurgical industries ."
2732,TN,Finnish Talvivaara Mining Co HEL : TLV1V said Thursday it had picked BofA Merrill Lynch and JPMorgan NYSE : JPM as joint bookrunners of its planned issue of convertible notes worth up to EUR250m USD332m .


In [22]:
noun_aux_verb_df['Class'].value_counts()

Class
FP    1311
TN    1054
TP     319
FN      50
Name: count, dtype: int64

In [ ]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_columns', 40)
# pd.set_option('display.max_rows', None)

In [23]:
filt_nav = noun_aux_verb_df['Class'] == "TP"
tp_nav_df = noun_aux_verb_df[filt_nav]
len(tp_nav_df)

319

In [24]:
tp_nav_df

,Class,Sentence
0,TP,"Known as Post Bank , the concept would see Fidelity Bank rolling out 75 offices in Ghana Post premises , to provide financial services to the people ."
1,TP,"Via the plant , the Belgian company will feed 8.4 MW of electricity into the distribution grid and will be able to deliver up to 14 MW of heat ."
2,TP,Castecka said the town hall would hold talks with other investors interested in the zone .
3,TP,"Jun. 14 , 2009 ( AOL Weblogs delivered by Newstex ) -- Looks like the E71 is about to be upstaged as Nokia 's ( NYSE : NOK ) premier business-class smartphone -- someone in Espoo 's just hit the corporate YouTube account with this promo video for an E72 ."
4,TP,Honkarakenne Oyj - a world-leading manufacturer of genuine wooden homes - will be sponsoring Finnish crosscountry skier Virpi Kuitunen for the next three years .
...,...,...
314,TP,The company anticipates its turnover for the whole 2010 to surpass that of the previous year when it was EUR 67.1 million .
315,TP,"Master of Mayawas jointly developed by Nokia Siemens Networks and UFA - FremantleMedia , and will be actively advertised by Maxis in the end of May 2007 ."
316,TP,Construction is scheduled to start in April-June 2007 and to be completed in early 2008 .
317,TP,"Pulkovo park will be ready in 2016 , its first stage of 23,000 sq. m. will be finished in the first quarter of 2010 ."


In [25]:
filt_nav = noun_aux_verb_df['Class'] == "FN"
fn_nav_df = noun_aux_verb_df[filt_nav]
len(fn_nav_df)

50

In [26]:
fn_nav_df

,Class,Sentence
319,FN,"An additional amount , capped at EUR12m , is payable in cash upon the achievement of certain financial performance targets in 2007 ."
320,FN,"UPM said the move will lower net profit by x20ac 385 million US$ 520 million in the second quarter , mainly due to impairment charges ."
321,FN,The deliveries started in April 2006 and will be completed in 2007 .
322,FN,"Konecranes has previously communicated an estimated reduction of about 1,600 employees on group level in 2009 ."
323,FN,Finnish KCI Konecranes has raised its net sales growth estimate for 2006 from over 25 % to over 35 % .
324,FN,Elcoteq expects its net sales for the last quarter of 2010 to be on the level of the third quarter .
325,FN,"LLC , a voice and data management solution provider to wireless companies with operations worldwide , will be transferring its U.S. deployment operations to the Finnish mobile giant , which includes civil works and site acquisition services ."
326,FN,Stora Enso Oyj said its second-quarter result would fall by half compared with the same period in 2007 .
327,FN,The transactions would increase earnings per share in the first quarter by some EUR0 .28 .
328,FN,Alfa group will have 43.9 % of voting stock in the new company and Telenor 35.4 % with a free float of 20.7 % .
